# TD 1 — L'indice des prix à la consommation

**Analyse des données — L3 Économie**

Ce TD porte sur une question simple en apparence : *de combien les prix ont-ils augmenté en France entre 2021 et 2023 ?*

Vous allez découvrir que la réponse dépend d'un choix, celui des pondérations, et que les données publiées ne permettent pas de calculer les indices tels que le manuel les définit.

**Avant de commencer :** menu *Exécution → Tout exécuter*. Prenez ce réflexe à chaque ouverture.

> Les cellules marquées `TODO` sont à compléter. Chaque `________` se remplace par une seule expression. Après chaque cellule complétée, **lisez la sortie** : plusieurs contiennent un contrôle qui vous dira si votre réponse tient debout.

## Partie 1 — Récupérer les données

Eurostat publie l'indice des prix harmonisé (IPCH) sous deux formes distinctes :

| Jeu de données | Contenu |
|---|---|
| `prc_hicp_inw` | les **pondérations** de chaque poste, en parts pour mille |
| `prc_hicp_ainr` | les **indices annuels** de chaque poste |

Les postes suivent la nomenclature COICOP : `CP01` alimentation, `CP04` logement et énergie, `CP07` transports, et ainsi de suite jusqu'à `CP12`.

Ces fichiers se chargent **directement depuis une adresse web**, sans téléchargement. Le notebook reste ainsi reproductible : il s'exécutera à l'identique chez quelqu'un d'autre.

> **Avant de coder :** ouvrez le *databrowser* d'Eurostat et retrouvez ces deux jeux de données. Vous devez savoir à quoi ressemble la documentation d'une source avant de l'utiliser. C'est là que sont écrites les définitions, et vous en aurez besoin dès cette première partie.

In [ ]:
import pandas as pd

BASE = "https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/"
OPT  = "?format=SDMX-CSV&compressed=false"

# Chargement direct depuis l'API : aucun telechargement, aucun televersement.
# Le notebook s'executera a l'identique chez votre binome.
ponderations = pd.read_csv(BASE + "prc_hicp_inw" + OPT)

print(ponderations.shape)
print(ponderations.columns.tolist())
ponderations.head()

In [ ]:
# TODO : chargez de la meme facon les indices annuels (prc_hicp_ainr)
indices = pd.read_csv(________)

print(indices.shape)
print(indices.columns.tolist())
indices.head()

**Question 1.** Combien de lignes chaque fichier contient-il ? De quoi une ligne est-elle la description, dans l'un et dans l'autre ? Répondez en une phrase par fichier.

*Votre réponse :*

### Deux fichiers, deux vocabulaires

Relisez les deux listes de colonnes que vous venez d'afficher. Elles ne sont pas identiques, et les différences ne sont pas cosmétiques.

La cellule suivante vous fait constater la première.

In [ ]:
# TODO : reperez, dans chaque liste de colonnes affichee ci-dessus,
#        le nom de la colonne qui designe les postes de consommation.

COL_POSTE_PONDER = "________"   # dans ponderations
COL_POSTE_INDICE = "________"   # dans indices

print("postes, ponderations :", ponderations[COL_POSTE_PONDER].nunique(), "modalites")
print("postes, indices      :", indices[COL_POSTE_INDICE].nunique(), "modalites")

Deux noms différents, et deux nombres de modalités différents. Retournez dans le *databrowser* : les deux jeux de données figurent-ils dans le même dossier, sous la même version de la nomenclature ?

**Notez votre constat ici.** Vous en aurez besoin à la question 3, et c'est la raison pour laquelle il ne faut pas se contenter de renommer la colonne pour faire tourner le code.

*Votre constat :*

### Le code de l'ensemble

Chaque fichier contient, en plus des douze postes, une ligne pour **l'ensemble de la consommation**. Son code n'est pas le même des deux côtés.

Trouvez-le. La cellule suivante teste vos deux hypothèses.

In [ ]:
# TODO : quel code designe l'ensemble dans chaque fichier ?
#        Indice : listez les modalites presentes, ou lisez le databrowser.

ENSEMBLE_PONDER = "________"
ENSEMBLE_INDICE = "________"

print(ENSEMBLE_PONDER, "dans les ponderations :",
      ENSEMBLE_PONDER in set(ponderations[COL_POSTE_PONDER]))
print(ENSEMBLE_INDICE, "dans les indices      :",
      ENSEMBLE_INDICE in set(indices[COL_POSTE_INDICE]))

Les deux lignes doivent afficher `True`. Si l'une affiche `False`, le code que vous avez écrit n'existe pas dans ce fichier. Cherchez encore avant de continuer : toute la partie 2 en dépend.

## Partie 2 — Reconstituer l'indice d'ensemble

L'indice d'ensemble est censé être la moyenne des indices de poste, pondérée par les parts budgétaires.

$$ I_{\text{ensemble}} = \frac{\sum_k w_k \, I_k}{\sum_k w_k} $$

Vous allez le recalculer pour la France en 2023, puis le comparer à la valeur publiée.

**Un piège, avant d'écrire la moindre ligne.** Le fichier des indices contient une colonne `unit`. Le même poste, pour le même pays et la même année, y apparaît **plusieurs fois** : une fois par unité de mesure. Sans filtre sur cette colonne, vous obtiendrez un tableau de la bonne taille et des valeurs fausses, sans aucun message d'erreur.

Regardez d'abord ce que contient cette colonne.

In [ ]:
print(indices["unit"].unique())

Celle qui nous intéresse est l'**indice moyen annuel**. Relevez son code, il sert à la cellule suivante.

In [ ]:
POSTES = ["CP0" + str(i) for i in range(1, 10)] + ["CP10", "CP11", "CP12"]


def extraire(df, colonne_poste, geo, annee, postes=POSTES, unite=None):
    """Renvoie un dictionnaire {poste: valeur} pour un pays et une annee."""
    d = df[(df["geo"] == geo)
           & (df["TIME_PERIOD"] == annee)
           & (df[colonne_poste].isin(postes))]
    if unite is not None:
        d = d[d["unit"] == unite]
    return dict(zip(d[colonne_poste], d["OBS_VALUE"]))


# Le fichier des ponderations n'a pas de colonne unit : rien a filtrer.
poids_2023 = extraire(ponderations, COL_POSTE_PONDER, "FR", 2023)

# TODO : le fichier des indices en a une. Passez le code releve ci-dessus.
indice_2023 = extraire(indices, COL_POSTE_INDICE, "FR", 2023, unite=________)

print("postes de ponderation  :", len(poids_2023))
print("postes d'indice        :", len(indice_2023))
print("somme des ponderations :", round(sum(poids_2023.values()), 2))

**Trois contrôles à lire avant de continuer.**

Douze postes de pondération. Douze postes d'indice. Et une somme des pondérations égale à **1 000**, puisqu'elles sont exprimées en parts pour mille.

Si la somme ne vaut pas 1 000, un poste manque ou un poste est compté deux fois. Et si vous obtenez douze indices sans avoir filtré sur `unit`, la longueur est bonne et les valeurs ne le sont pas : c'est exactement le genre d'erreur que ce cours vous apprend à débusquer.

In [ ]:
# TODO : calculez la moyenne ponderee
numerateur   = 0
denominateur = 0

for poste in poids_2023:
    numerateur   += ________
    denominateur += ________

indice_reconstitue = numerateur / denominateur
print("Indice reconstitue :", round(indice_reconstitue, 2))

In [ ]:
# TODO : recuperez l'indice d'ensemble publie pour la France en 2023.
#        Attention au code de l'ensemble dans CE fichier.
indice_publie = extraire(indices, COL_POSTE_INDICE, "FR", 2023,
                         postes=[________], unite=________)[________]

print("Indice publie :", round(indice_publie, 2))
print("Ecart         :", round(indice_reconstitue - indice_publie, 2), "point")
print("Ecart relatif : %.2f %%" % (100 * (indice_reconstitue / indice_publie - 1)))

**Question 2.** L'écart n'est pas nul. Ce n'est pas une erreur de votre part. De quel ordre de grandeur est-il, en point d'indice et en pourcentage ?

- Comparez-le à la précision que l'INSEE annonce sur le taux d'inflation, de l'ordre du dixième de point. Cet écart est-il négligeable ?

**Question 3.** Cherchez dans la documentation méthodologique de l'IPCH pourquoi la moyenne pondérée des sous-indices ne redonne pas exactement l'indice publié. Citez au moins **deux** raisons.

- Puis ajoutez-en une troisième, que vous n'avez pas trouvée dans la documentation mais dans ce notebook : relisez votre constat de la partie 1.
- Laquelle des trois vous paraît la plus susceptible de produire un écart de cette taille ? Une raison peut être vraie et quantitativement négligeable.

*Votre réponse :*

### La réponse à la question de la séance

Avant d'aller plus loin, répondons à la question posée en ouverture. Elle porte sur l'indice publié, pas sur votre reconstitution.

In [ ]:
# TODO : recuperez l'indice d'ensemble publie pour 2021 et 2022
total_2021 = ________
total_2022 = ________
total_2023 = indice_publie

print("Indice d'ensemble France :", total_2021, total_2022, total_2023)
print("Hausse 2021 -> 2022 : %.2f %%" % (100 * (total_2022 / total_2021 - 1)))
print("Hausse 2022 -> 2023 : %.2f %%" % (100 * (total_2023 / total_2022 - 1)))
print("Hausse 2021 -> 2023 : %.2f %%" % (100 * (total_2023 / total_2021 - 1)))

**Trois choses à écrire ici**, avant de passer à la partie 3.

- De combien les prix ont-ils augmenté en France entre 2021 et 2023 ?
- Les trois indices valent moins de 100, alors que les prix augmentent. Pourquoi ? Cherchez la **base** de l'indice dans la documentation du jeu de données.
- La hausse 2021–2023 n'est pas la somme des deux hausses annuelles. Écrivez la relation exacte entre les trois.

*Vos réponses :*

## Partie 3 — Le choix des pondérations

Les pondérations changent chaque année : les ménages ne consomment pas en 2023 ce qu'ils consommaient en 2021.

Vous allez mesurer ce que ce choix implique. Recalculez l'indice 2023, une fois avec les pondérations de 2023, une fois avec celles de 2021. **Les indices restent ceux de 2023 dans les deux cas** : seule la pondération change.

In [ ]:
# TODO : construisez poids_2021 et indice_2021 sur le meme modele qu'en partie 2
poids_2021  = ________
indice_2021 = ________


def indice_pondere(indices_poste, poids):
    """Moyenne des indices de poste, ponderee par les parts budgetaires."""
    # TODO : reprenez la logique de la partie 2
    numerateur   = 0
    denominateur = 0
    for poste in poids:
        numerateur   += ________
        denominateur += ________
    return numerateur / denominateur


avec_poids_2023 = indice_pondere(indice_2023, poids_2023)
avec_poids_2021 = indice_pondere(indice_2023, poids_2021)

print("Indice 2023, ponderations 2023 :", round(avec_poids_2023, 2))
print("Indice 2023, ponderations 2021 :", round(avec_poids_2021, 2))
print("Ecart                          :", round(avec_poids_2021 - avec_poids_2023, 2), "point")

**Question 4.** Lequel des deux est le plus élevé ? Rapprochez ce résultat de la distinction entre indice de Laspeyres et indice de Paasche vue en cours.

- De combien les deux versions diffèrent-elles ? Comparez cet écart à celui de la question 2, puis à l'arrondi avec lequel un indice est publié.
- Le calcul que vous venez de faire est-il vraiment une comparaison entre Laspeyres et Paasche ? Regardez quels indices entrent dans les deux versions avant de répondre.
- Que peut-on honnêtement conclure d'un écart de cette taille ?

*Votre réponse :*

### Pondérations et prix relatifs

On s'attend à ce que les ménages réduisent la part des postes dont le prix augmente le plus. Le tableau suivant permet de le vérifier, poste par poste.

In [ ]:
variation = {p: poids_2023[p] - poids_2021[p] for p in poids_2021 if p in poids_2023}

print("%-6s %8s %8s %10s %10s" % ("poste", "I 2021", "I 2023", "hausse %", "dW pmille"))
for p, v in sorted(variation.items(), key=lambda x: x[1]):
    hausse = 100 * (indice_2023[p] / indice_2021[p] - 1)
    print("%-6s %8.1f %8.1f %+10.1f %+10.1f"
          % (p, indice_2021[p], indice_2023[p], hausse, v))

In [ ]:
# TODO : mesurez la relation plutot que de la lire a l'oeil.
#        Deux Series indexees par poste, puis leur correlation.
postes_communs = sorted(variation)

hausses = pd.Series({p: ________ for p in postes_communs})
poids_d = pd.Series({p: ________ for p in postes_communs})

print("correlation sur les 12 postes :", round(hausses.corr(poids_d), 3))

**Question 5.** Identifiez les postes dont la pondération a le plus reculé entre 2021 et 2023. Sont-ce aussi ceux dont le prix a le plus augmenté ? Est-ce une coïncidence ?

Procédez dans cet ordre.

- Nommez les **deux postes dont la pondération a le plus reculé**. Leur prix a-t-il beaucoup augmenté ?
- Nommez les **deux postes dont la pondération a le plus progressé**. Que consomme-t-on davantage en 2023 qu'en 2021, et pourquoi ? Pensez à ce qui s'est passé en 2021.
- La corrélation que vous venez de calculer confirme-t-elle l'hypothèse de substitution ?
- Recalculez-la dans la cellule suivante, en retirant les deux postes de la question précédente. Que devient-elle ?

> **Attention à la formulation.** Douze points ne permettent d'établir aucune causalité. Le choix de retirer deux d'entre eux est lui-même une décision, qui doit être justifiée et annoncée. Écrivez ce que vous observez, pas ce que vous auriez aimé observer.

*Votre réponse :*

In [ ]:
# TODO : la meme correlation, sans les deux postes de rattrapage
sans_rebond = [p for p in postes_communs if p not in (________, ________)]

print("en retirant ces deux postes :",
      round(hausses[sans_rebond].corr(poids_d[sans_rebond]), 3))

**Question 6.** Un ménage dont les revenus sont indexés sur l'indice préfère-t-il la version à pondérations 2021 ou 2023 ? Et l'organisme qui verse cette indexation ?

- Répondez d'abord sur le principe.
- Puis chiffrez l'enjeu réel sur ces deux années, pour une pension de 1 500 euros par mois. Le principe et l'enjeu sont deux questions différentes.

*Votre réponse :*

## Partie 4 — La même hausse de prix, deux pays

Les pondérations diffèrent d'un pays à l'autre : la part de l'énergie dans le budget des ménages n'est pas la même en France et en Hongrie.

Reprenez les **indices français** de 2023, et appliquez-leur les **pondérations d'un autre pays**. Le choc de prix est identique ; l'inflation mesurée ne l'est pas.

In [ ]:
# TODO : recuperez les ponderations 2023 de chacun de ces pays
liste_pays = ["HU", "BG", "EE", "DE"]

print("FR : indices francais, ponderations FR -> %.2f"
      % indice_pondere(indice_2023, poids_2023))

for pays in liste_pays:
    poids_pays = ________
    print("%s : indices francais, ponderations %s -> %.2f"
          % (pays, pays, indice_pondere(indice_2023, poids_pays)))

**Question 7.** Commentez l'écart. Que dit-il sur la comparabilité internationale des taux d'inflation ?

- Chiffrez l'écart entre le plus petit et le plus grand des cinq résultats, en point et en pourcentage, puis comparez-le à celui de la question 2.
- Le classement des cinq pays est-il aléatoire ? Que savez-vous de la part de l'alimentation et de l'énergie dans le budget des ménages en Bulgarie et en Allemagne ?

**Question 8.** L'IPCH est dit « harmonisé ». Au vu de ce que vous venez de calculer, sur quoi porte exactement l'harmonisation, et sur quoi ne porte-t-elle pas ?

*Vos réponses :*

## Pour la séance 2

Vérifiez que votre notebook s'exécute **entièrement de haut en bas sans erreur**, puis déposez-le selon les modalités indiquées par votre chargé de TD.

Un notebook qui ne passe pas de haut en bas est un notebook faux : il contient des variables calculées dans un ordre que personne ne pourra reproduire.

Aucune note n'est attribuée. En revanche, les questions de diagnostic du contrôle continu porteront sur les erreurs les plus fréquentes rencontrées ici.